# Lesson 03 - Agentic Design Patterns

Sa araling ito, susuriin natin ang tatlong pangunahing disenyo na pattern para sa paggawa ng epektibong AI agents:

1. **Malinaw na Tagubilin para sa Agent** — Paggawa ng tumpak, papel na naglalarawang prompts na gumagabay sa kilos ng agent
2. **Nakaayos na Output gamit ang Pydantic Models** — Pagtiyak na ang mga agent ay nagbabalik ng predictable, validated na data
3. **Single Responsibility Agents** — Pagdidisenyo ng fokus na mga agent na bawat isa ay mahusay sa isang bagay

I-aapply natin ang bawat pattern sa isang **travel destination recommender** na senaryo, na unti-unting bumubuo ng sistema na makakapag-suggest ng mga destinasyon, magche-check ng availability, at humawak ng logistics.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: Malinaw na Mga Tagubilin para sa Ahente

Ang pinakakapana-panabik na pattern ay ang pinaka-simple rin: pagsulat ng malinaw, detalyadong mga tagubilin para sa iyong ahente.

Ang magagandang tagubilin ay naglalarawan:
- **Sino** ang ahente (persona at tono)
- **Ano** ang dapat gawin nito (mga hakbang-hakbang na responsibilidad)
- **Paano** ito dapat kumilos (mga limitasyon at estilo)

Sa ibaba, lumilikha tayo ng isang travel concierge agent na may malinaw na mga tagubilin na humuhubog sa bawat tugon na ginagawa nito.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Istrakturadong Output gamit ang mga Pydantic Models

Ang malayang teksto ay kapaki-pakinabang para sa usapan, ngunit nangangailangan ang mga downstream system ng istrakturadong datos.
Sa pamamagitan ng pagsasama ng **mga Pydantic model** sa isang **tool function**, maaari nating:

- Tukuyin ang eksaktong iskema para sa output ng ahente
- Awtomatikong beripikahin ang mga tugon
- I-integrate ang mga resulta ng ahente sa lohika ng aplikasyon nang maaasahan

Ipinakikilala rin namin ang isang tool na nagbabalik ng mga detalye ng destinasyon upang maipundar ng ahente ang mga rekomendasyon nito sa totoong datos.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Mga Ahente na May Iisang Responsibilidad

Ang mga kumplikadong gawain ay nakikinabang sa paghahati-hati ng trabaho sa maraming nakatuon na ahente, bawat isa ay may iisang responsibilidad:

- Isang **Eksperto sa Destinasyon** na may kaalaman tungkol sa mga lugar at availability
- Isang **Tagaplano ng Logistika** na humahawak sa mga flight, hotel, at itineraryo

Ito ay kahalintulad ng prinsipyo sa software engineering na *separation of concerns* — mas madali ang pagsubok, pagpapanatili, at pagpapabuti ng bawat ahente nang magkahiwalay.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Buod

Sa araling ito inilapat namin ang tatlong agentic design patterns sa isang travel recommender scenario:

| Pattern | Pangunahing Ideya | Benepisyo |
|---|---|---|
| **Clear Instructions** | Tukuyin ang persona, mga responsibilidad, at mga limitasyon mula sa simula | Konsistent, on-brand na pag-uugali ng ahente |
| **Structured Output** | Gumamit ng Pydantic models bilang format ng sagot | Na-validate, machine-readable na mga resulta |
| **Single Responsibility** | Bigyan ang bawat ahente ng isang pokus na trabaho | Mas madaling subukan, panatilihin, at pag-combine |

Ang mga pattern na ito ay natural na nagkakakombina — maaari mong pagsamahin ang clear instructions sa structured output sa loob ng isang single-responsibility agent upang makabuo ng matibay, production-ready na mga sistema.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang AI na serbisyo sa pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kami sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa kanyang likas na wika ang dapat ituring na opisyal na sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
